In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.7 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import time
import platform
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
DRIVE_INPUT_PATH = Path("/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss")

INPUT_PATH = DRIVE_INPUT_PATH
OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input path:", INPUT_PATH)
print("Output dir:", OUTPUT_DIR)

Input path: /content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet
Output dir: /content/drive/MyDrive/tablewise/artifacts_new/faiss


In [ ]:
needed_cols = [
    "restaurant_id",
    "name",
    "country",
    "region",
    "province",
    "city",
    "city_filled",
    "city_filled_norm",
    "city_source",
    "address",
    "latitude",
    "longitude",
    "rating",
    "price_level",
    "price_bucket",
    "top_tags_text",
    "top_tags_norm_list",
    "meals_text",
    "meals_norm_list",
    "features_text",
    "features_norm_list",
    "special_diets_text",
    "special_diets_norm_list",
    "popularity_detailed",
    "popularity_rank_num",
    "popularity_total_num",
    "popularity_score",
    "profile_quality_score",
    "profile_text",
    "short_profile",
]

available_cols = pd.read_parquet(INPUT_PATH, columns=None).columns.tolist()
cols_to_load = [c for c in needed_cols if c in available_cols]

df = pd.read_parquet(INPUT_PATH, columns=cols_to_load)

print("Loaded shape:", df.shape)
print("Columns loaded:", df.columns.tolist())

display(df.head(3))

Loaded shape: (1033881, 30)
Columns loaded: ['restaurant_id', 'name', 'country', 'region', 'province', 'city', 'city_filled', 'city_filled_norm', 'city_source', 'address', 'latitude', 'longitude', 'rating', 'price_level', 'price_bucket', 'top_tags_text', 'top_tags_norm_list', 'meals_text', 'meals_norm_list', 'features_text', 'features_norm_list', 'special_diets_text', 'special_diets_norm_list', 'popularity_detailed', 'popularity_rank_num', 'popularity_total_num', 'popularity_score', 'profile_quality_score', 'profile_text', 'short_profile']


,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,profile_text,short_profile
0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu...","Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French"
1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur...","Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats"
2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch...","Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European"


In [ ]:
required_cols = ["restaurant_id", "name", "profile_text", "short_profile"]

for col in required_cols:
    assert col in df.columns, f"Lipsește coloana obligatorie: {col}"

print("Coloanele obligatorii există.")

print("Rows:", len(df))
print("Duplicate restaurant_id:", df["restaurant_id"].duplicated().sum())
print("Missing profile_text:", df["profile_text"].isna().sum())
print("Missing short_profile:", df["short_profile"].isna().sum())

display(df[["restaurant_id", "name", "city_filled", "price_bucket", "profile_text"]].head(3))

Coloanele obligatorii există.
Rows: 1033881
Duplicate restaurant_id: 0
Missing profile_text: 0
Missing short_profile: 0


,restaurant_id,name,city_filled,price_bucket,profile_text
0,g10001637-d10002227,Le 147,Saint-Jouvent,cheap,"Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,g10001637-d14975787,Le Saint Jouvent,Saint-Jouvent,cheap,"Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,g10002858-d4586832,Au Bout du Pont,Rivarennes,cheap,"Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."


In [ ]:
df_embed = df.copy()

df_embed["profile_text"] = (
    df_embed["profile_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_embed["short_profile"] = (
    df_embed["short_profile"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

before = len(df_embed)

df_embed = df_embed[
    (df_embed["restaurant_id"].notna())
    & (df_embed["name"].notna())
    & (df_embed["profile_text"] != "")
    & (df_embed["short_profile"] != "")
].copy().reset_index(drop=True)

after = len(df_embed)

print(f"Eliminate profile goale sau invalide: {before - after}")
print("Shape final înainte de sampling:", df_embed.shape)

Eliminate profile goale sau invalide: 0
Shape final înainte de sampling: (1033881, 30)


In [ ]:
bad_profile_mask = df_embed["profile_text"].str.contains(
    r"\b(nan|none|null)\b",
    case=False,
    regex=True,
    na=False,
)

print("Profile cu nan/none/null textual:", int(bad_profile_mask.sum()))

if bad_profile_mask.sum() > 0:
    display(df_embed.loc[bad_profile_mask, ["restaurant_id", "name", "profile_text"]].head(20))

print("\nLungime profile_text:")
display(df_embed["profile_text"].str.len().describe())

print("\nTop city_filled:")
if "city_filled" in df_embed.columns:
    display(df_embed["city_filled"].value_counts(dropna=False).head(20))

print("\nPrice bucket:")
if "price_bucket" in df_embed.columns:
    display(df_embed["price_bucket"].value_counts(dropna=False))

Profile cu nan/none/null textual: 83


,restaurant_id,name,profile_text
6848,g1080243-d14013161,O Nan House,"O Nan House is a restaurant located in Cergy-Pontoise, Val-d'Oise, Ile-de-France, France. Address: 221 rue Henri Dunant, 95300, Cergy-Pontoise France. Average rating: 3.0 out of 5. Cuisines and tags: Fast food. Meals served: Lunch, Dinner. Popularity: #11 of 19 Restaurants in Cergy-Pontoise."
22654,g1571443-d4794280,Chez Yves,"Chez Yves is a restaurant located in Cantobre, Aveyron, Occitanie, France. Address: None, Cantobre France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Breakfast, Lunch, Dinner. Features: Reservations, Outdoor ..."
26459,g1857768-d14980264,Pizza Nan's,"Pizza Nan's is a restaurant located in Saint-Marcellin, Isere, Auvergne-Rhone-Alpes, France. Address: 6 B avenue de Romans, 38160 Saint-Marcellin France. Average rating: 4.0 out of 5. Meals served: Lunch, Dinner. Popularity: #10 of 11 Restaurants in Saint-Marcellin."
36312,g187130-d13412711,Nan Kebab,"Nan Kebab is a restaurant located in Tours, Indre-et-Loire, Centre-Val de Loire, France. Address: 14 avenue Andre Maginot, 37100, Tours France. Average rating: 2.5 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Middle Eastern. Popularity: #414 of 432 Resta..."
38022,g187143-d12378286,Deli nan Hassan,"Deli nan Hassan is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 90 rue Battant, 25000, Besancon France. Average rating: 3.5 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Middle Eastern. Meals served: Dinner. Featu..."
38106,g187143-d15707269,Deli' Nan 2,"Deli' Nan 2 is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 31 rue d Arenes, 25000, Besancon France."
38711,g187147-d1024076,Qiao Jiang Nan,"Qiao Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 43 rue de Provence, 75009 Paris France. Average rating: 4.0 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Chinese, Asian. Meals served: Dinner, Lunch. Features: Seating, ..."
46247,g187147-d19378717,Nouilles De Jiang Nan,"Nouilles De Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 21 rue Mademoiselle, 75015 Paris France. Average rating: 4.5 out of 5. Meals served: Lunch, Dinner. Features: Reservations. Popularity: #10388 of 15476 Restaurants in Paris."
51958,g187147-d6102671,Jiang Nan,"Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 10 rue de Mazagran metro Bonne-Nouvelle ou Strasbourg St-denis, 75010 Paris France. Average rating: 4.5 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, Chinese, Asian, Vietnamese...."
57473,g187153-d12945759,Le Cheese Nan,"Le Cheese Nan is a restaurant located in Montpellier, Herault, Occitanie, France. Address: 17 Rue de Verdun, 2 rue Aristide Olivier, 34000, Montpellier France. Average rating: 2.0 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Indian, Fast food, Turkish. M..."



Lungime profile_text:


,profile_text
count,1.033881e+06
mean,3.652525e+02
std,9.830401e+01
min,8.500000e+01
25%,2.990000e+02
50%,3.790000e+02
75%,4.220000e+02
max,9.840000e+02



Top city_filled:


,count
city_filled,
Paris,18129
Rome,12603
Madrid,12134
Barcelona,10285
Milan,8382
Prague,6035
Lisbon,5261
Vienna,4571
Amsterdam,4325



Price bucket:


,count
price_bucket,
mid,372580
unknown,262779
expensive,233064
cheap,165458


In [ ]:

bad_profile_mask = df_embed["profile_text"].str.contains(
    r"\b(nan|none|null)\b",
    case=False,
    regex=True,
    na=False,
)

print("Profile cu nan/none/null textual inainte de drop:", int(bad_profile_mask.sum()))

if bad_profile_mask.sum() > 0:
    print("\nExemple profile problematice:")
    display(df_embed.loc[bad_profile_mask, ["restaurant_id", "name", "profile_text"]].head(20))

    before = len(df_embed)

    df_embed = df_embed.loc[~bad_profile_mask].copy().reset_index(drop=True)

    after = len(df_embed)

    print(f"\nProfile eliminate: {before - after}")
    print("Shape dupa drop:", df_embed.shape)

bad_profile_mask_after = df_embed["profile_text"].str.contains(
    r"\b(nan|none|null)\b",
    case=False,
    regex=True,
    na=False,
)

print("\nProfile cu nan/none/null textual dupa drop:", int(bad_profile_mask_after.sum()))

print("\nLungime profile_text:")
display(df_embed["profile_text"].str.len().describe())

print("\nTop city_filled:")
if "city_filled" in df_embed.columns:
    display(df_embed["city_filled"].value_counts(dropna=False).head(20))

print("\nPrice bucket:")
if "price_bucket" in df_embed.columns:
    display(df_embed["price_bucket"].value_counts(dropna=False))

Profile cu nan/none/null textual inainte de drop: 83

Exemple profile problematice:


,restaurant_id,name,profile_text
6848,g1080243-d14013161,O Nan House,"O Nan House is a restaurant located in Cergy-Pontoise, Val-d'Oise, Ile-de-France, France. Address: 221 rue Henri Dunant, 95300, Cergy-Pontoise France. Average rating: 3.0 out of 5. Cuisines and tags: Fast food. Meals served: Lunch, Dinner. Popularity: #11 of 19 Restaurants in Cergy-Pontoise."
22654,g1571443-d4794280,Chez Yves,"Chez Yves is a restaurant located in Cantobre, Aveyron, Occitanie, France. Address: None, Cantobre France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Breakfast, Lunch, Dinner. Features: Reservations, Outdoor ..."
26459,g1857768-d14980264,Pizza Nan's,"Pizza Nan's is a restaurant located in Saint-Marcellin, Isere, Auvergne-Rhone-Alpes, France. Address: 6 B avenue de Romans, 38160 Saint-Marcellin France. Average rating: 4.0 out of 5. Meals served: Lunch, Dinner. Popularity: #10 of 11 Restaurants in Saint-Marcellin."
36312,g187130-d13412711,Nan Kebab,"Nan Kebab is a restaurant located in Tours, Indre-et-Loire, Centre-Val de Loire, France. Address: 14 avenue Andre Maginot, 37100, Tours France. Average rating: 2.5 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Middle Eastern. Popularity: #414 of 432 Resta..."
38022,g187143-d12378286,Deli nan Hassan,"Deli nan Hassan is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 90 rue Battant, 25000, Besancon France. Average rating: 3.5 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Middle Eastern. Meals served: Dinner. Featu..."
38106,g187143-d15707269,Deli' Nan 2,"Deli' Nan 2 is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 31 rue d Arenes, 25000, Besancon France."
38711,g187147-d1024076,Qiao Jiang Nan,"Qiao Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 43 rue de Provence, 75009 Paris France. Average rating: 4.0 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Chinese, Asian. Meals served: Dinner, Lunch. Features: Seating, ..."
46247,g187147-d19378717,Nouilles De Jiang Nan,"Nouilles De Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 21 rue Mademoiselle, 75015 Paris France. Average rating: 4.5 out of 5. Meals served: Lunch, Dinner. Features: Reservations. Popularity: #10388 of 15476 Restaurants in Paris."
51958,g187147-d6102671,Jiang Nan,"Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 10 rue de Mazagran metro Bonne-Nouvelle ou Strasbourg St-denis, 75010 Paris France. Average rating: 4.5 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, Chinese, Asian, Vietnamese...."
57473,g187153-d12945759,Le Cheese Nan,"Le Cheese Nan is a restaurant located in Montpellier, Herault, Occitanie, France. Address: 17 Rue de Verdun, 2 rue Aristide Olivier, 34000, Montpellier France. Average rating: 2.0 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Indian, Fast food, Turkish. M..."



Profile eliminate: 83
Shape dupa drop: (1033798, 30)

Profile cu nan/none/null textual dupa drop: 0

Lungime profile_text:


,profile_text
count,1.033798e+06
mean,3.652535e+02
std,9.830565e+01
min,8.500000e+01
25%,2.990000e+02
50%,3.790000e+02
75%,4.220000e+02
max,9.840000e+02



Top city_filled:


,count
city_filled,
Paris,18126
Rome,12603
Madrid,12133
Barcelona,10283
Milan,8381
Prague,6035
Lisbon,5261
Vienna,4571
Amsterdam,4325



Price bucket:


,count
price_bucket,
mid,372551
unknown,262760
expensive,233056
cheap,165431


In [ ]:
USE_SAMPLE = False
SAMPLE_SIZE = 100_000

if USE_SAMPLE:
    if len(df_embed) > SAMPLE_SIZE:
        df_embed = df_embed.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    else:
        df_embed = df_embed.reset_index(drop=True)
else:
    df_embed = df_embed.reset_index(drop=True)

print("Shape final pentru embeddings:", df_embed.shape)

display(df_embed[["restaurant_id", "name", "city_filled", "price_bucket", "short_profile"]].head(5))

Shape final pentru embeddings: (1033798, 30)


,restaurant_id,name,city_filled,price_bucket,short_profile
0,g10001637-d10002227,Le 147,Saint-Jouvent,cheap,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French"
1,g10001637-d14975787,Le Saint Jouvent,Saint-Jouvent,cheap,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats"
2,g10002858-d4586832,Au Bout du Pont,Rivarennes,cheap,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European"
3,g10002986-d3510044,Le Relais de Naiade,Lacelle,cheap,"Le Relais de Naiade | Lacelle, France | rating=4.0 | price=cheap | tags=Cheap Eats, French"
4,g10022428-d9767191,Relais Du MontSeigne,Saint-Laurent-de-Levezou,mid,"Relais Du MontSeigne | Saint-Laurent-de-Levezou, France | rating=4.5 | price=mid | tags=Mid-range, French"


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Model loaded.")
print("Max sequence length:", model.max_seq_length)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Max sequence length: 256


In [ ]:
texts = df_embed["profile_text"].tolist()

print("Număr texte:", len(texts))
print("\nExemplu profile_text:")
print(texts[0][:1000])

Număr texte: 1033798

Exemplu profile_text:
Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Features: Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service. Popularity: #1 of 2 Restaurants in Saint-Jouvent.


In [ ]:
test_embedding = model.encode(
    ["test"],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embedding_dim = int(test_embedding.shape[1])
estimated_bytes = len(df_embed) * embedding_dim * 4
estimated_gb = estimated_bytes / (1024 ** 3)

print("Embedding dim:", embedding_dim)
print(f"Estimated embeddings memory: {estimated_gb:.2f} GB")

Embedding dim: 384
Estimated embeddings memory: 1.48 GB


In [ ]:
BATCH_SIZE = 512
CHUNK_SIZE = 300_000
all_embeddings = []

start_time = time.time()

for start_idx in range(0, len(texts), CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, len(texts))
    chunk_texts = texts[start_idx:end_idx]

    print(f"Encoding chunk {start_idx:,} - {end_idx:,} / {len(texts):,}")

    chunk_embeddings = model.encode(
        chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    all_embeddings.append(chunk_embeddings)

    del chunk_embeddings
    gc.collect()

embeddings = np.vstack(all_embeddings).astype("float32")

elapsed = time.time() - start_time

print("Embeddings shape:", embeddings.shape)
print(f"Encoding time: {elapsed / 60:.2f} minutes")

Encoding chunk 0 - 300,000 / 1,033,798


Batches:   0%|          | 0/586 [00:00<?, ?it/s]

Encoding chunk 300,000 - 600,000 / 1,033,798


Batches:   0%|          | 0/586 [00:00<?, ?it/s]

Encoding chunk 600,000 - 900,000 / 1,033,798


Batches:   0%|          | 0/586 [00:00<?, ?it/s]

Encoding chunk 900,000 - 1,033,798 / 1,033,798


Batches:   0%|          | 0/262 [00:00<?, ?it/s]

Embeddings shape: (1033798, 384)
Encoding time: 11.12 minutes


In [ ]:
assert embeddings.shape[0] == len(df_embed), "Numarul de embeddings nu corespunde cu numarul de restaurante"
assert embeddings.shape[1] == embedding_dim, "Dimensiunea embeddings nu corespunde"

print("Embeddings dtype:", embeddings.dtype)
print("Embeddings shape:", embeddings.shape)

norms = np.linalg.norm(embeddings[:1000], axis=1)

print("Norm mean first 1000:", norms.mean())
print("Norm min first 1000:", norms.min())
print("Norm max first 1000:", norms.max())

assert np.all(np.isfinite(embeddings)), "Exista NaN/Inf in embeddings"

Embeddings dtype: float32
Embeddings shape: (1033798, 384)
Norm mean first 1000: 1.0
Norm min first 1000: 0.99999994
Norm max first 1000: 1.0000001


In [ ]:
EMBEDDINGS_PATH = OUTPUT_DIR / "restaurant_embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

print("Saved embeddings:", EMBEDDINGS_PATH)
print("File size GB:", EMBEDDINGS_PATH.stat().st_size / (1024 ** 3))

Saved embeddings: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy
File size GB: 1.4788600206375122


In [ ]:
import faiss


embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

print("Index is trained:", index.is_trained)
print("Adding embeddings to FAISS...")

index.add(embeddings)

print("FAISS ntotal:", index.ntotal)

assert index.ntotal == len(df_embed), "FAISS index size nu corespunde cu df_embed."

Index is trained: True
Adding embeddings to FAISS...
FAISS ntotal: 1033798


In [ ]:
FAISS_INDEX_PATH = OUTPUT_DIR / "restaurant_faiss.index"

faiss.write_index(index, str(FAISS_INDEX_PATH))

print("Saved FAISS index:", FAISS_INDEX_PATH)
print("File size GB:", FAISS_INDEX_PATH.stat().st_size / (1024 ** 3))

Saved FAISS index: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index
File size GB: 1.4788599433377385


In [ ]:
mapping_cols = [
    "restaurant_id",
    "name",
    "country",
    "region",
    "province",
    "city",
    "city_filled",
    "city_filled_norm",
    "city_source",
    "address",
    "latitude",
    "longitude",
    "rating",
    "price_level",
    "price_bucket",
    "top_tags_text",
    "top_tags_norm_list",
    "meals_text",
    "meals_norm_list",
    "features_text",
    "features_norm_list",
    "special_diets_text",
    "special_diets_norm_list",
    "popularity_detailed",
    "popularity_rank_num",
    "popularity_total_num",
    "popularity_score",
    "profile_quality_score",
    "short_profile",
    "profile_text",
]

mapping_cols = [c for c in mapping_cols if c in df_embed.columns]

index_mapping = df_embed[mapping_cols].copy().reset_index(drop=True)
index_mapping.insert(0, "faiss_idx", np.arange(len(index_mapping), dtype=np.int64))

print("Mapping shape:", index_mapping.shape)
display(index_mapping.head(3))

assert len(index_mapping) == index.ntotal, "Mapping size nu corespunde cu FAISS index."
assert index_mapping["faiss_idx"].is_unique, "faiss_idx nu este unic."

Mapping shape: (1033798, 31)


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."


In [ ]:
MAPPING_PATH = OUTPUT_DIR / "restaurant_index_mapping.parquet"
MAPPING_CSV_PATH = OUTPUT_DIR / "restaurant_index_mapping.csv"

index_mapping.to_parquet(MAPPING_PATH, index=False)
index_mapping.to_csv(MAPPING_CSV_PATH, index=False)

print("Saved mapping parquet:", MAPPING_PATH)
print("Saved mapping csv:", MAPPING_CSV_PATH)

Saved mapping parquet: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
Saved mapping csv: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.csv


In [ ]:
EMBEDDING_DATA_PATH = OUTPUT_DIR / "restaurants_for_embeddings.parquet"

df_embed.to_parquet(EMBEDDING_DATA_PATH, index=False)

print("Saved restaurants used for embeddings:", EMBEDDING_DATA_PATH)
print("Rows df_embed:", len(df_embed))
print("Rows embeddings:", embeddings.shape[0])

assert len(df_embed) == embeddings.shape[0]

Saved restaurants used for embeddings: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet
Rows df_embed: 1033798
Rows embeddings: 1033798


In [ ]:
METADATA_PATH = OUTPUT_DIR / "metadata.json"

metadata = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "input_path": str(INPUT_PATH),
    "embedding_data_path": str(EMBEDDING_DATA_PATH),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "num_restaurants": int(len(df_embed)),
    "embedding_dim": int(embedding_dim),
    "normalized_embeddings": True,
    "faiss_index_type": "IndexFlatIP",
    "use_sample": bool(USE_SAMPLE),
    "sample_size": int(SAMPLE_SIZE) if USE_SAMPLE else None,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "faiss_index_path": str(FAISS_INDEX_PATH),
    "mapping_path": str(MAPPING_PATH),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved metadata:", METADATA_PATH)
metadata

Saved metadata: /content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json


{'created_at': '2026-05-04T16:00:35.667690Z',
 'input_path': '/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet',
 'embedding_data_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'normalized_embeddings': True,
 'faiss_index_type': 'IndexFlatIP',
 'use_sample': False,
 'sample_size': None,
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'python_version': '3.12.13',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2'}

In [ ]:
loaded_embeddings = np.load(EMBEDDINGS_PATH)
loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))
loaded_mapping = pd.read_parquet(MAPPING_PATH)

print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded FAISS ntotal:", loaded_index.ntotal)
print("Loaded mapping shape:", loaded_mapping.shape)

assert loaded_embeddings.shape == embeddings.shape
assert loaded_index.ntotal == index.ntotal
assert len(loaded_mapping) == len(index_mapping)

print("Reload check OK.")

Loaded embeddings shape: (1033798, 384)
Loaded FAISS ntotal: 1033798
Loaded mapping shape: (1033798, 31)
Reload check OK.


In [ ]:
def semantic_search(query, model, index, mapping_df, top_k=10):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    result_indices = indices[0]
    result_scores = scores[0]

    results = mapping_df.iloc[result_indices].copy()
    results["semantic_score"] = result_scores

    return results.reset_index(drop=True)

In [ ]:
query = "cheap italian restaurant in rome"

results = semantic_search(
    query=query,
    model=model,
    index=index,
    mapping_df=index_mapping,
    top_k=10,
)

cols_to_show = [
    "faiss_idx",
    "restaurant_id",
    "name",
    "country",
    "city_filled",
    "rating",
    "price_bucket",
    "top_tags_text",
    "short_profile",
    "semantic_score",
]

cols_to_show = [c for c in cols_to_show if c in results.columns]

display(results[cols_to_show])

,faiss_idx,restaurant_id,name,country,city_filled,rating,price_bucket,top_tags_text,short_profile,semantic_score
0,665507,g187791-d13373510,Sale & Pepe Roma,Italy,Rome,4.5,mid,"Mid-range, Italian, Pizza, Mediterranean","Sale & Pepe Roma | Rome, Italy | rating=4.5 | price=mid | tags=Mid-range, Italian, Pizza, Mediterranean",0.759475
1,673678,g187791-d7247248,Pizzeria Romaest,Italy,Rome,3.0,cheap,"Cheap Eats, Bakeries, Italian, Pizza","Pizzeria Romaest | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Bakeries, Italian, Pizza",0.756497
2,667132,g187791-d1725968,Pasta and Social International House,Italy,Rome,4.0,cheap,"Cheap Eats, Italian, Pizza, Mediterranean","Pasta and Social International House | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Mediterranean",0.755544
3,664392,g187791-d11992847,The New,Italy,Rome,4.0,cheap,"Cheap Eats, Italian, Bar, Pizza","The New | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Bar, Pizza",0.754694
4,666429,g187791-d15210430,Baked Restaurant & Drinks,Italy,Rome,3.5,mid,"Mid-range, Italian, Seafood, Mediterranean","Baked Restaurant & Drinks | Rome, Italy | rating=3.5 | price=mid | tags=Mid-range, Italian, Seafood, Mediterranean",0.749335
5,667829,g187791-d1903542,Wanted,Italy,Rome,3.0,mid,"Mid-range, Italian, Pizza, Mediterranean","Wanted | Rome, Italy | rating=3.0 | price=mid | tags=Mid-range, Italian, Pizza, Mediterranean",0.748630
6,670716,g187791-d3493500,Chinese Restaurant Internazionale,Italy,Rome,4.0,cheap,"Cheap Eats, Chinese, Asian, Vegetarian Friendly","Chinese Restaurant Internazionale | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Chinese, Asian, Vegetarian Friendly",0.748223
7,665670,g187791-d13783786,Movie Restaurant,Italy,Rome,4.5,expensive,"Mid-range, Italian, Mediterranean, Romana","Movie Restaurant | Rome, Italy | rating=4.5 | price=expensive | tags=Mid-range, Italian, Mediterranean, Romana",0.745827
8,663837,g187791-d10818976,Hosteria Cacio & Pepe,Italy,Rome,4.0,cheap,"Cheap Eats, Italian, Romana, Lazio","Hosteria Cacio & Pepe | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Romana, Lazio",0.742029
9,666742,g187791-d15675546,Wiki Wiki Eat,Italy,Rome,3.5,cheap,"Cheap Eats, Italian, Mediterranean, Romana","Wiki Wiki Eat | Rome, Italy | rating=3.5 | price=cheap | tags=Cheap Eats, Italian, Mediterranean, Romana",0.740454


In [ ]:
def pretty_print_results(query, results, top_n=5):
    print(f"\nQuery: {query}")
    print("-" * 120)

    for i, (_, row) in enumerate(results.head(top_n).iterrows(), start=1):
        name = row.get("name", "Unknown")
        city = row.get("city_filled", "Unknown")
        country = row.get("country", "Unknown")
        rating = row.get("rating", "Unknown")
        price = row.get("price_bucket", "Unknown")
        score = row.get("semantic_score", np.nan)
        short_profile = row.get("short_profile", "")

        print(f"[{i}] {name}")
        print(f"    Location: {city}, {country}")
        print(f"    Rating: {rating}")
        print(f"    Price bucket: {price}")
        print(f"    Semantic score: {score:.4f}")
        print(f"    {short_profile}")
        print()

In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "highly rated seafood restaurant in lisbon",
    "romantic dinner in paris",
    "vegetarian restaurant in berlin",
    "breakfast cafe in athens",
    "fine dining restaurant in milan",
    "popular tapas place in madrid",
    "family friendly restaurant in barcelona",
]

for q in test_queries:
    results = semantic_search(
        query=q,
        model=model,
        index=index,
        mapping_df=index_mapping,
        top_k=5,
    )

    pretty_print_results(q, results, top_n=5)
    print("=" * 120)


Query: cheap italian restaurant in rome
------------------------------------------------------------------------------------------------------------------------
[1] Sale & Pepe Roma
    Location: Rome, Italy
    Rating: 4.5
    Price bucket: mid
    Semantic score: 0.7595
    Sale & Pepe Roma | Rome, Italy | rating=4.5 | price=mid | tags=Mid-range, Italian, Pizza, Mediterranean

[2] Pizzeria Romaest
    Location: Rome, Italy
    Rating: 3.0
    Price bucket: cheap
    Semantic score: 0.7565
    Pizzeria Romaest | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Bakeries, Italian, Pizza

[3] Pasta and Social International House
    Location: Rome, Italy
    Rating: 4.0
    Price bucket: cheap
    Semantic score: 0.7555
    Pasta and Social International House | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Mediterranean

[4] The New
    Location: Rome, Italy
    Rating: 4.0
    Price bucket: cheap
    Semantic score: 0.7547
    The New | Rome, Italy

In [ ]:
city_test_queries = [
    ("cheap italian restaurant in rome", "rome"),
    ("seafood restaurant in lisbon", "lisbon"),
    ("romantic dinner in paris", "paris"),
    ("vegetarian restaurant in berlin", "berlin"),
    ("fine dining restaurant in milan", "milan"),
    ("tapas place in madrid", "madrid"),
]

diagnostics = []

for query, expected_city in city_test_queries:
    results = semantic_search(
        query=query,
        model=model,
        index=index,
        mapping_df=index_mapping,
        top_k=10,
    )

    if "city_filled_norm" in results.columns:
        city_match_count = (results["city_filled_norm"] == expected_city).sum()
    else:
        city_match_count = results["city_filled"].astype(str).str.lower().eq(expected_city).sum()

    diagnostics.append({
        "query": query,
        "expected_city": expected_city,
        "matches_in_top_10": int(city_match_count),
        "top_result_city": results.iloc[0].get("city_filled", None),
        "top_result_name": results.iloc[0].get("name", None),
    })

diagnostic_df = pd.DataFrame(diagnostics)
display(diagnostic_df)

,query,expected_city,matches_in_top_10,top_result_city,top_result_name
0,cheap italian restaurant in rome,rome,10,Rome,Sale & Pepe Roma
1,seafood restaurant in lisbon,lisbon,10,Lisbon,Seafood
2,romantic dinner in paris,paris,6,Franqueville-Saint-Pierre,La Romantica
3,vegetarian restaurant in berlin,berlin,0,Nantes,Berlin 1989
4,fine dining restaurant in milan,milan,7,Milan,EAT Restaurant
5,tapas place in madrid,madrid,10,Madrid,Tapa Tapa


In [ ]:
price_test_queries = [
    ("cheap restaurant in rome", "cheap"),
    ("fine dining restaurant in paris", "expensive"),
    ("mid range restaurant in madrid", "mid"),
]

price_diagnostics = []

for query, expected_price in price_test_queries:
    results = semantic_search(
        query=query,
        model=model,
        index=index,
        mapping_df=index_mapping,
        top_k=10,
    )

    if "price_bucket" in results.columns:
        price_match_count = (results["price_bucket"] == expected_price).sum()
    else:
        price_match_count = 0

    price_diagnostics.append({
        "query": query,
        "expected_price_bucket": expected_price,
        "matches_in_top_10": int(price_match_count),
        "top_result_price": results.iloc[0].get("price_bucket", None),
        "top_result_name": results.iloc[0].get("name", None),
    })

price_diagnostic_df = pd.DataFrame(price_diagnostics)
display(price_diagnostic_df)

,query,expected_price_bucket,matches_in_top_10,top_result_price,top_result_name
0,cheap restaurant in rome,cheap,5,cheap,Wiki Wiki Eat
1,fine dining restaurant in paris,expensive,5,cheap,The Place To Eat
2,mid range restaurant in madrid,mid,8,mid,Best Restaurant
